# Plant Disease Detection: train on Kaggle (free GPU)

**Before running (one time):**
1. Right side panel -> **Session options / Settings**: set **Accelerator = GPU (T4 or P100)** and **Internet = On** (Internet needs a verified phone number; it is required to download the ImageNet weights).
2. **Add Input** -> search **PlantVillage Dataset** (by Abdallah Ali, 38 classes) -> **+** to add it.
3. Click **Run All**.

When it finishes, download **`results.zip`** from the **Output** panel on the right (`/kaggle/working`), unzip it into your project folder on your PC (so `models/` and `outputs/` sit next to `app.py`), and run `streamlit run app.py`.

In [ ]:
# ---- Settings you may change -------------------------------------------
MAX_PER_CLASS = 1000   # images per class (faster). Use None for the full dataset (slower, a bit more accurate)
CNN_EPOCHS = 20
VIT_EPOCHS = 20
HEAD_EPOCHS = 5        # transfer learning: head only
FINETUNE_EPOCHS = 8    # transfer learning: fine-tune top layers
BATCH_SIZE = 32


In [ ]:
import os, glob, tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "No GPU! Turn on Accelerator = GPU in the notebook settings, then run again."

# find the folder holding the 38 class folders (the 'color' version)
cands = [d for d in glob.glob("/kaggle/input/**/color", recursive=True) if os.path.isdir(d)]
assert cands, "Dataset not found: use Add Input -> PlantVillage Dataset"
DATA_DIR = cands[0]
print("Dataset folder:", DATA_DIR, "| classes:", len(os.listdir(DATA_DIR)))

In [ ]:
%cd /kaggle/working
!mkdir -p plant_disease_detection/src

In [ ]:
%%writefile /kaggle/working/plant_disease_detection/requirements.txt
# Use Python 3.10 - 3.12 (check tensorflow.org/install for the versions your TF release supports).
tensorflow>=2.16,<3        # brings Keras 3; models use the .keras format
keras>=3.0
numpy>=1.26
pandas>=2.0
scikit-learn>=1.3
matplotlib>=3.7
Pillow>=10.0
streamlit>=1.35
# Optional: only for scripts/download_dataset.py
# kagglehub


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/app.py
"""Streamlit app: Plant Disease Detector with an interactive care guide.

Run from the project root:   streamlit run app.py
"""
from __future__ import annotations

import hashlib
import io
import json
from datetime import datetime

import pandas as pd
import streamlit as st
from PIL import Image, ImageOps, UnidentifiedImageError

from src import ui
from src.config import (CLASS_NAMES_PATH, COMPARISON_DIR, DISPLAY_NAMES, MODEL_NAMES, MODELS_DIR,
                        OUTPUTS_DIR, model_path)
from src.disease_info import CHEMICAL_NOTE, DISCLAIMER, DISEASE_TYPES, get_info
from src.labels import parse_label, pretty_label

st.set_page_config(page_title="Plant Disease Intelligence & Care Planner", page_icon="🌿", layout="wide")
st.markdown(ui.CSS, unsafe_allow_html=True)

ss = st.session_state
if "history" not in ss:
    ss["history"] = []
    ss["history_keys"] = set()

IMAGE_TYPES = ["jpg", "jpeg", "png", "bmp", "webp"]


# ------------------------------------------------------------------ data helpers
def available_models() -> list[str]:
    names = [n for n in MODEL_NAMES if model_path(n).exists()]
    extras = sorted(p.stem for p in MODELS_DIR.glob("*.keras") if p.stem not in MODEL_NAMES)
    return names + extras


def load_class_names() -> list[str]:
    if CLASS_NAMES_PATH.exists():
        return json.loads(CLASS_NAMES_PATH.read_text())
    return []


@st.cache_resource(show_spinner="Loading model...")
def get_predictor(name: str):
    from src.predict import Predictor  # lazy import: TensorFlow is slow to load
    return Predictor(model_path(name))


@st.cache_data(show_spinner=False, max_entries=512)
def predict_cached(model_name: str, image_bytes: bytes) -> list[dict]:
    """Top-5 predictions. Cached so ticking a checkbox never re-runs the model."""
    image = Image.open(io.BytesIO(image_bytes))
    image.load()
    return get_predictor(model_name).predict(image, top_k=5)


def label_of(p: dict) -> str:
    return f"{p['plant']} - {p['condition']}"


def record_history(item: dict, model_name: str, preds: list[dict]) -> None:
    key = (item["id"], model_name)
    if key in ss["history_keys"]:
        return
    ss["history_keys"].add(key)
    top = preds[0]
    ss["history"].append({
        "Time": datetime.now().strftime("%H:%M:%S"),
        "Image": item["name"],
        "Model": DISPLAY_NAMES.get(model_name, model_name),
        "Plant": top["plant"],
        "Result": top["condition"],
        "Confidence %": round(top["confidence"] * 100, 1),
    })


def bullets(items: list[str]) -> None:
    if items:
        st.markdown("\n".join(f"- {t}" for t in items))


# ------------------------------------------------------------------ guide renderers
def render_guide(info: dict, key_prefix: str) -> None:
    """Full interactive care guide (tabs + tick-off action plan)."""
    chips = ui.type_chip(info["type"]) + ("" if info["healthy"] else ui.severity_chip(info["severity"]))
    st.markdown(f"#### {info['name']}")
    st.markdown(chips, unsafe_allow_html=True)
    if not info["known"]:
        st.info("No detailed guide is stored for this class yet - ask a local agronomist.")

    if info["healthy"]:
        t_over, t_care = st.tabs(["📋 Overview", "🌱 Keep it healthy"])
        with t_over:
            st.write(info["summary"])
            bullets(info["symptoms"])
        with t_care:
            st.markdown("**This week**")
            bullets(info["immediate"])
            st.markdown("**Long term**")
            bullets(info["prevention"])
        return

    t_over, t_plan, t_org, t_chem, t_prev = st.tabs(
        ["📋 Overview & Cause", "✅ Action Plan", "🌿 Organic & Cultural", "🧪 Chemical Options", "🛡️ Prevention"]
    )
    with t_over:
        st.write(info["summary"])
        st.markdown(ui.kv("Causative Agent", info["agent"]) + ui.kv("Environmental Conditions Favoured", info["favours"]), unsafe_allow_html=True)
        st.markdown("**Diagnostic Symptoms**")
        bullets(info["symptoms"])
        if info.get("spread_note"):
            kind = "bad" if info["severity"] in ("High", "Critical") else "warn"
            st.markdown(ui.callout(f"{info['severity']} severity. {info['spread_note']}", kind), unsafe_allow_html=True)
    with t_plan:
        steps = info["immediate"]
        st.caption("Tick each step as you complete it:")
        progress = st.empty()
        keys = []
        for i, step in enumerate(steps):
            key = f"{key_prefix}_step_{i}"
            keys.append(key)
            st.checkbox(step, key=key)
        done = sum(bool(ss.get(k)) for k in keys)
        progress.progress(done / max(len(steps), 1), text=f"{done} of {len(steps)} steps done")
        if steps and done == len(steps):
            st.success("All immediate steps done. Keep monitoring plants over the next week.")
    with t_org:
        bullets(info["organic"] or ["No organic option is listed for this problem."])
    with t_chem:
        st.markdown(ui.callout(CHEMICAL_NOTE, "warn"), unsafe_allow_html=True)
        bullets(info["chemical"] or ["No chemical option is listed for this problem."])
    with t_prev:
        bullets(info["prevention"])


def render_compact(info: dict) -> None:
    """Short read-only version for the disease library (no widgets)."""
    st.markdown(ui.type_chip(info["type"]) + ("" if info["healthy"] else ui.severity_chip(info["severity"])),
                unsafe_allow_html=True)
    st.write(info["summary"])
    if info["agent"]:
        st.markdown(ui.kv("Cause", info["agent"]), unsafe_allow_html=True)
    if info["symptoms"]:
        st.markdown("**Symptoms**")
        bullets(info["symptoms"])
    st.markdown("**What to do**" if not info["healthy"] else "**Keep it healthy**")
    bullets(info["immediate"])
    if info["prevention"]:
        st.markdown("**Prevention**")
        bullets(info["prevention"])


# ------------------------------------------------------------------ diagnose helpers
def build_items(files) -> list[dict]:
    items = []
    for f in files:
        data = f.getvalue()
        try:
            img = Image.open(io.BytesIO(data))
            img.load()
        except (UnidentifiedImageError, OSError):
            st.warning(f"Skipped '{getattr(f, 'name', 'file')}': it could not be read as an image.")
            continue
        items.append({
            "id": hashlib.md5(data).hexdigest()[:10],
            "name": getattr(f, "name", None) or "camera photo",
            "bytes": data,
            "image": ImageOps.exif_transpose(img).convert("RGB"),
        })
    return items


def render_compare(item: dict, models: list[str]) -> None:
    rows, top_classes = [], []
    with st.spinner("Running inference across all models..."):
        for m in models:
            p = predict_cached(m, item["bytes"])
            top_classes.append(p[0]["class_name"])
            rows.append({
                "Architecture": DISPLAY_NAMES.get(m, m),
                "Top Prediction": label_of(p[0]),
                "Confidence %": round(p[0]["confidence"] * 100, 1),
                "2nd Choice": label_of(p[1]) if len(p) > 1 else "",
            })
    st.dataframe(pd.DataFrame(rows), hide_index=True)
    if len(set(top_classes)) == 1:
        st.success(f"High Agreement: All {len(models)} models independently agree on {label_of(parse_dict(top_classes[0]))}!")
    else:
        st.warning("Model Divergence Detected: The architectures disagree. If the photo includes fingers or busy backgrounds, crop to just the leaf blade.")


def parse_dict(class_name: str) -> dict:
    plant, condition, _ = parse_label(class_name)
    return {"plant": plant, "condition": condition}


def render_detail(item: dict, model_name: str, models: list[str], top_k: int, threshold: float) -> None:
    preds = item["preds"]
    top = preds[0]
    left, right = st.columns([1, 1.2], gap="large")
    with left:
        st.image(item["image"], caption=f"📸 {item['name']}", use_container_width=True)
        notes = ui.quality_notes(item["image"])
        if notes:
            with st.expander("⚠️ Image Quality Diagnostic"):
                for note in notes:
                    st.caption("• " + note)
    with right:
        st.markdown(ui.result_card_html(top, get_info(top["class_name"])), unsafe_allow_html=True)
        for kind, text in ui.confidence_messages(preds, threshold):
            st.markdown(ui.callout(text, kind), unsafe_allow_html=True)
        st.markdown("##### Prediction Probability Distribution")
        st.markdown(ui.bars_html(preds[:top_k]), unsafe_allow_html=True)

    st.divider()
    st.subheader("📋 Agronomic Care Plan & Disease Treatment")
    options = [p["class_name"] for p in preds[:top_k]]
    labels = {p["class_name"]: f"{label_of(p)} ({p['confidence'] * 100:.1f}%)" for p in preds[:top_k]}
    chosen = st.radio("Inspect treatment protocol for candidate:", options, format_func=labels.get, horizontal=True,
                      key=f"sel_{item['id']}_{model_name}_{top_k}")
    info = get_info(chosen)
    render_guide(info, key_prefix=f"{item['id']}_{chosen}")

    st.divider()
    c1, c2 = st.columns([1, 1])
    with c1:
        st.download_button(
            "⬇️ Export Clinical Diagnosis Report (Markdown)",
            key=f"dl_{item['id']}_{chosen}",
            data=ui.report_markdown(item["name"], DISPLAY_NAMES.get(model_name, model_name), preds[:top_k], info),
            file_name=f"plant_report_{item['id']}.md",
            mime="text/markdown",
        )
    with c2:
        if len(models) > 1:
            compare = st.toggle("🔬 Cross-Validate with All Models (ViT, ResNet50, MobileNet, CNN)", key=f"cmp_{item['id']}")
        else:
            compare = False
    if compare:
        render_compare(item, models)
    st.caption(DISCLAIMER)


# ------------------------------------------------------------------ Header & Sidebar
st.markdown(
    """<div class="hero">
    <div class="hero-badges">
        <span class="hero-pill glow">🌱 AI AgroDiagnostic Suite</span>
        <span class="hero-pill">⚡ Vision Transformer (ViT) & Deep CNNs</span>
        <span class="hero-pill">38 Disease Classes</span>
    </div>
    <h1>🌿 Plant Disease Intelligence & Care Planner</h1>
    <p>Automated visual crop pathology powered by Deep Learning. Detect fungal, bacterial, viral, and pest infections from leaf imagery, review confidence metrics, and follow agronomist-approved treatment workflows.</p>
</div>""",
    unsafe_allow_html=True,
)

models = available_models()
class_names = load_class_names()

with st.sidebar:
    st.header("⚙️ Model & Diagnostics")
    if models:
        # Default to ViT if available, otherwise ResNet50
        def_idx = models.index("vit") if "vit" in models else (models.index("resnet50") if "resnet50" in models else 0)
        model_name = st.selectbox("Active Architecture", models, format_func=lambda n: DISPLAY_NAMES.get(n, n), index=def_idx)
    else:
        model_name = None
        st.warning("No trained model found in the models folder.")

    top_k = st.slider("Predictions to show", 1, 5, 3)
    threshold = st.slider("Warn if confidence is below (%)", 30, 95, 65)

    st.divider()
    st.subheader("💡 Best Photography Practices")
    st.caption(
        "• **One leaf per photo:** Frame leaf centered against a plain or ground background.\n"
        "• **Avoid fingers/hands:** Skin tones and shadows can bias prediction.\n"
        "• **Sharp focus:** Tap on leaf lesion to ensure micro-textures are crisp.\n"
        "• **Good lighting:** Avoid harsh direct glare or severe under-exposure."
    )
    st.divider()
    st.caption(f"Session history: **{len(ss['history'])} diagnoses**")

tab_diag, tab_lib, tab_perf, tab_about = st.tabs(
    ["🔍 Plant Diagnosis", "📚 Disease Library", "📊 Model Performance & Benchmarks", "ℹ️ Architecture & About"]
)

# ------------------------------------------------------------------ Diagnose
with tab_diag:
    if model_name is None or not class_names:
        st.info("Train at least one model first (see README.md), then reload this page. "
                "The app needs `models/*.keras` and `outputs/class_names.json`.")
    else:
        # Quick Stat Ribbon
        active_name = DISPLAY_NAMES.get(model_name, model_name).split(" (")[0]
        st.markdown(
            f"""<div class="stat-grid">
            <div class="stat-card"><span class="stat-label">Active Architecture</span><span class="stat-val">{active_name}</span></div>
            <div class="stat-card"><span class="stat-label">Knowledge Base</span><span class="stat-val">{len(class_names)} Diseases</span></div>
            <div class="stat-card"><span class="stat-label">Confidence Threshold</span><span class="stat-val">{threshold}%</span></div>
            <div class="stat-card"><span class="stat-label">Diagnoses Completed</span><span class="stat-val">{len(ss['history'])}</span></div>
        </div>""",
            unsafe_allow_html=True,
        )

        mode = st.radio("Image Source", ["Upload photos", "Use camera"], horizontal=True)
        if mode == "Upload photos":
            files = st.file_uploader("Upload leaf photos (single or batch)", type=IMAGE_TYPES, accept_multiple_files=True)
        else:
            shot = st.camera_input("Capture live leaf photo")
            files = [shot] if shot else []

        if not files:
            st.markdown(
                '<div class="empty"><div class="big">🍃</div><b>Upload or capture a leaf photo to begin</b><br>'
                'You will receive instant pathogen diagnosis, confidence probability distribution, and a customized step-by-step treatment plan.</div>',
                unsafe_allow_html=True,
            )
        else:
            items = build_items(files)
            if items:
                with st.spinner("Analyzing plant pathology with Deep Neural Network..."):
                    for it in items:
                        try:
                            it["preds"] = predict_cached(model_name, it["bytes"])
                        except Exception as exc:
                            st.error(f"Prediction failed: {exc}")
                            st.stop()
                        record_history(it, model_name, it["preds"])

                if len(items) > 1:
                    st.markdown(f"##### Batch Diagnosis Overview ({len(items)} images)")
                    st.dataframe(pd.DataFrame([{
                        "Image": it["name"],
                        "Plant": it["preds"][0]["plant"],
                        "Diagnosis": it["preds"][0]["condition"],
                        "Confidence %": round(it["preds"][0]["confidence"] * 100, 1),
                        "Health Status": "Healthy" if it["preds"][0]["healthy"] else "Disease Detected",
                    } for it in items]), hide_index=True)
                    pick = st.selectbox("Select image to inspect details:", range(len(items)),
                                        format_func=lambda i: f"{i + 1}. {items[i]['name']}")
                    item = items[pick]
                else:
                    item = items[0]
                render_detail(item, model_name, models, top_k, threshold)

    if ss["history"]:
        st.divider()
        with st.expander(f"🕑 Session Diagnosis Log ({len(ss['history'])})"):
            hist = pd.DataFrame(ss["history"])
            st.dataframe(hist, hide_index=True)
            h1, h2 = st.columns(2)
            h1.download_button("⬇️ Export Session Log (CSV)", data=hist.to_csv(index=False),
                               file_name="diagnosis_history.csv", mime="text/csv")
            if h2.button("Clear session log"):
                ss["history"] = []
                ss["history_keys"] = set()
                st.rerun()

# ------------------------------------------------------------------ Library
with tab_lib:
    if not class_names:
        st.info("The disease library needs `outputs/class_names.json` (created when you train).")
    else:
        entries = []
        for cls in class_names:
            plant, cond, _ = parse_label(cls)
            entries.append((cls, plant, cond, get_info(cls)))
        c1, c2, c3 = st.columns([2, 1.5, 1.5])
        query = c1.text_input("🔎 Search Pathology Database", placeholder="e.g. tomato, blight, virus, rust, healthy")
        plant_sel = c2.multiselect("Filter by Crop", sorted({e[1] for e in entries}))
        type_sel = c3.multiselect("Filter by Pathogen Type", list(DISEASE_TYPES))
        q = query.strip().lower()
        shown = [e for e in entries
                 if (not q or q in f"{e[1]} {e[2]} {e[3]['name']} {e[3]['type']} {e[3]['summary']}".lower())
                 and (not plant_sel or e[1] in plant_sel) and (not type_sel or e[3]["type"] in type_sel)]
        st.caption(f"Showing **{len(shown)} of {len(entries)}** plant disease conditions.")
        for cls, plant, cond, info in shown:
            with st.expander(f"{'✅' if info['healthy'] else '🦠'} {plant} — {cond}"):
                render_compact(info)

# ------------------------------------------------------------------ Performance
with tab_perf:
    table_path = COMPARISON_DIR / "metrics_comparison.csv"
    if not table_path.exists():
        st.info("No comparison yet. Run `python -m src.evaluate` after training.")
    else:
        df = pd.read_csv(table_path)
        m1, m2, m3 = st.columns(3)
        m1.metric("Highest Test Accuracy", df.loc[df["Test accuracy"].idxmax(), "Model"].split(" (")[0],
                  f"{df['Test accuracy'].max() * 100:.1f}%")
        m2.metric("Most Compact Model", df.loc[df["Size (MB)"].idxmin(), "Model"].split(" (")[0],
                  f"{df['Size (MB)'].min():.1f} MB")
        m3.metric("Fastest Training", df.loc[df["Train time (min)"].idxmin(), "Model"].split(" (")[0],
                  f"{df['Train time (min)'].min():.1f} min")
        st.dataframe(df, hide_index=True)
        metric = st.selectbox("Compare models across benchmark metric:", list(df.select_dtypes("number").columns))
        st.bar_chart(df.set_index("Model")[metric])
        for img_name, caption in (("metrics_comparison.png", "Comparative Test Metrics"),
                                  ("training_curves_comparison.png", "Validation Loss & Accuracy Curves")):
            f = COMPARISON_DIR / img_name
            if f.exists():
                st.image(str(f), caption=caption)

        trained = [n for n in MODEL_NAMES if (OUTPUTS_DIR / n / "metrics.json").exists()]
        if trained:
            st.subheader("Model-Specific Deep Dive")
            pick = st.selectbox("Inspect Architecture", trained, format_func=lambda n: DISPLAY_NAMES[n], key="perf_model")
            cm_kind = st.radio("Confusion Matrix Visualization", ["Row-normalised", "Raw Counts"], horizontal=True)
            c1, c2 = st.columns(2)
            curves = OUTPUTS_DIR / pick / "training_curves.png"
            cm = OUTPUTS_DIR / pick / ("confusion_matrix_normalized.png" if cm_kind == "Row-normalised"
                                       else "confusion_matrix.png")
            if curves.exists():
                c1.image(str(curves), caption="Training & Validation Curves")
            if cm.exists():
                c2.image(str(cm), caption="Pathology Confusion Matrix")
            rep_path = OUTPUTS_DIR / pick / "classification_report.csv"
            if rep_path.exists():
                rep = pd.read_csv(rep_path, index_col=0)
                rep = rep.loc[rep.index.isin(class_names)] if class_names else rep
                n = st.slider("Show weakest N classes by F1-Score:", 3, max(len(rep), 3), min(10, max(len(rep), 3)))
                weakest = rep["f1-score"].sort_values().head(n)
                weakest.index = [pretty_label(i) for i in weakest.index]
                st.bar_chart(weakest)
                with st.expander("Full Per-Class Precision / Recall / F1-Score Table"):
                    st.dataframe(rep)

# ------------------------------------------------------------------ About
with tab_about:
    st.markdown(
        """
### Deep Learning Architecture & Dataset Overview

**Dataset** — **PlantVillage**: ~54,000 labeled images spanning 38 classes (14 crop species: Apple, Blueberry, Cherry, Corn, Grape, Orange, Peach, Bell Pepper, Potato, Raspberry, Soybean, Squash, Strawberry, Tomato).

**Supported Architectures**
- **Vision Transformer (ViT)**: Pure self-attention vision architecture dividing images into 16x16 non-overlapping patches with learnable positional embeddings, Multi-Head Attention, and GELU MLP blocks.
- **Custom CNN**: 4 convolution blocks with batch-norm, pooling, and dropout, trained from scratch.
- **MobileNetV2**: Efficient inverted residual backbone pretrained on ImageNet; ultra-light and fast for edge deployment.
- **ResNet50**: Deep residual backbone pretrained on ImageNet; superior feature extraction capacity.

**Pipeline Specifications**
- Input dimensions: 224x224 RGB
- In-model data augmentation (flips, rotations, zoom, translation, contrast) active during training
- Integrated normalization layers allowing raw image ingestion
- Stratified 70/15/15 train/val/test splits with balanced class weighting

**Agronomic Care Guide**
- Integrated evidence-based intervention steps, cultural practices, organic biological controls, and chemical fungicide/bactericide recommendations.
"""
    )
    st.warning(DISCLAIMER)
    if class_names:
        with st.expander("Explore all 38 supported crop & disease classes"):
            st.write(sorted(pretty_label(c) for c in class_names))


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/__init__.py
"""Plant Disease Detection: CNN vs transfer learning on PlantVillage."""


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/config.py
"""Central configuration for the Plant Disease Detection project."""
from __future__ import annotations

from pathlib import Path

# ---------------------------------------------------------------- paths
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "PlantVillage"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SPLIT_PATH = OUTPUTS_DIR / "split.json"
CLASS_NAMES_PATH = OUTPUTS_DIR / "class_names.json"
COMPARISON_DIR = OUTPUTS_DIR / "comparison"

# ---------------------------------------------------------------- data
IMG_SIZE = (224, 224)
IMG_SHAPE = (*IMG_SIZE, 3)
BATCH_SIZE = 32
SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

# train / validation / test fractions (must sum to 1)
SPLIT_FRACTIONS = (0.70, 0.15, 0.15)

# ---------------------------------------------------------------- training
CNN_EPOCHS = 20
HEAD_EPOCHS = 5           # transfer learning phase 1: frozen backbone
FINE_TUNE_EPOCHS = 8      # transfer learning phase 2: unfreeze top layers
FINE_TUNE_LAYERS = 30     # how many backbone layers to unfreeze in phase 2
LR_CNN = 1e-3
LR_HEAD = 1e-3
LR_FINE_TUNE = 1e-5

# ViT (Vision Transformer) parameters
VIT_PATCH_SIZE = 16
VIT_PROJECTION_DIM = 128
VIT_NUM_HEADS = 4
VIT_TRANSFORMER_LAYERS = 4
VIT_MLP_DIM = 256
VIT_DROPOUT = 0.1
VIT_EPOCHS = 20
LR_VIT = 1e-3

# ---------------------------------------------------------------- models
MODEL_NAMES = ("cnn", "mobilenetv2", "resnet50", "vit")
DISPLAY_NAMES = {
    "cnn": "Custom CNN (from scratch)",
    "mobilenetv2": "MobileNetV2 (transfer learning)",
    "resnet50": "ResNet50 (transfer learning)",
    "vit": "Vision Transformer (ViT from scratch)",
}


def model_path(name: str) -> Path:
    return MODELS_DIR / f"{name}.keras"


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/data_loader.py
"""tf.data input pipeline: decode -> resize -> batch -> prefetch.

Images stay in the raw 0-255 range on purpose: each model contains its own
normalisation layer (see models.py), so the saved models accept raw images and the
Streamlit app needs no model-specific preprocessing.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import tensorflow as tf

from .config import (CLASS_NAMES_PATH, IMG_SIZE, SEED, SPLIT_FRACTIONS, SPLIT_PATH)
from .data_utils import find_dataset_root, get_or_create_split

AUTOTUNE = tf.data.AUTOTUNE


def _decode(path: tf.Tensor, label: tf.Tensor):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)            # bilinear
    img = tf.cast(img, tf.float32)                  # keep 0-255
    img.set_shape((*IMG_SIZE, 3))
    return img, label


def make_dataset(root, paths, labels, batch_size: int, training: bool) -> tf.data.Dataset:
    full_paths = [str(Path(root) / p) for p in paths]
    ds = tf.data.Dataset.from_tensor_slices((full_paths, np.asarray(labels, dtype=np.int32)))
    if training:
        ds = ds.shuffle(len(full_paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_decode, num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size).prefetch(AUTOTUNE)


def prepare_data(data_dir, batch_size: int, max_per_class: int | None = None,
                 seed: int = SEED, resplit: bool = False) -> dict:
    """Build train/val/test datasets and persist the split + class names."""
    root = find_dataset_root(data_dir)
    split = get_or_create_split(root, SPLIT_PATH, SPLIT_FRACTIONS, seed, max_per_class, force=resplit)
    class_names = split["class_names"]
    CLASS_NAMES_PATH.parent.mkdir(parents=True, exist_ok=True)
    CLASS_NAMES_PATH.write_text(json.dumps(class_names, indent=2))

    root_used = Path(split["meta"]["root"])
    datasets = {
        name: make_dataset(root_used, split[name]["paths"], split[name]["labels"],
                           batch_size, training=(name == "train"))
        for name in ("train", "val", "test")
    }
    print(f"[data] root={root_used}")
    print(f"[data] {len(class_names)} classes | train={len(split['train']['paths'])} "
          f"val={len(split['val']['paths'])} test={len(split['test']['paths'])}")
    return {"class_names": class_names, "split": split, "root": root_used, **{f"{k}_ds": v for k, v in datasets.items()}}


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/data_utils.py
"""Framework-free dataset helpers: discovery, stratified splitting, class weights.

Kept separate from ``data_loader`` (which needs TensorFlow) so they can be
unit-tested and reused without a deep-learning install.
"""
from __future__ import annotations

import json
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from .config import IMAGE_EXTENSIONS


def find_dataset_root(data_dir: str | Path) -> Path:
    """Locate the folder that directly contains the class sub-folders.

    PlantVillage downloads come nested in different ways, e.g.
    ``PlantVillage/PlantVillage/<classes>`` or ``plantvillage dataset/color/<classes>``.
    """
    root = Path(data_dir)
    if not root.is_dir():
        raise FileNotFoundError(
            f"Dataset folder not found: {root}\n"
            "See README.md ('Get the dataset') for download instructions."
        )
    for _ in range(4):
        if (root / "color").is_dir():          # abdallahalidev/plantvillage-dataset layout
            root = root / "color"
            continue
        subdirs = sorted(d for d in root.iterdir() if d.is_dir() and not d.name.startswith("."))
        has_images = any(f.suffix.lower() in IMAGE_EXTENSIONS for f in root.iterdir() if f.is_file())
        if not has_images and len(subdirs) == 1:  # single wrapper folder -> descend
            root = subdirs[0]
            continue
        break
    return root


def list_images(root: str | Path) -> tuple[list[str], list[int], list[str]]:
    """Return (relative image paths, integer labels, sorted class names)."""
    root = Path(root)
    class_dirs = sorted(d for d in root.iterdir() if d.is_dir() and not d.name.startswith("."))
    paths: list[str] = []
    labels: list[int] = []
    class_names: list[str] = []
    for d in class_dirs:
        files = sorted(f for f in d.rglob("*") if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS)
        if not files:
            continue
        idx = len(class_names)
        class_names.append(d.name)
        for f in files:
            paths.append(f.relative_to(root).as_posix())
            labels.append(idx)
    if len(class_names) < 2:
        raise ValueError(f"Found {len(class_names)} class folder(s) with images under {root}; need at least 2.")
    return paths, labels, class_names


def limit_per_class(paths: list[str], labels: list[int], n: int, seed: int) -> tuple[list[str], list[int]]:
    """Randomly keep at most ``n`` images per class (for quick CPU runs)."""
    rng = random.Random(seed)
    by_class: dict[int, list[int]] = defaultdict(list)
    for i, y in enumerate(labels):
        by_class[y].append(i)
    keep: list[int] = []
    for y in sorted(by_class):
        idxs = by_class[y]
        keep.extend(sorted(rng.sample(idxs, n)) if len(idxs) > n else idxs)
    keep.sort()
    return [paths[i] for i in keep], [labels[i] for i in keep]


def stratified_split(paths, labels, fractions, seed) -> dict:
    """Stratified train/val/test split that keeps class proportions."""
    train_f, val_f, test_f = fractions
    if abs(train_f + val_f + test_f - 1.0) > 1e-6:
        raise ValueError("Split fractions must sum to 1.")
    idx = np.arange(len(paths))
    y = np.asarray(labels)
    idx_train, idx_rest = train_test_split(idx, train_size=train_f, stratify=y, random_state=seed)
    idx_val, idx_test = train_test_split(
        idx_rest, test_size=test_f / (val_f + test_f), stratify=y[idx_rest], random_state=seed
    )
    out = {}
    for name, ids in (("train", idx_train), ("val", idx_val), ("test", idx_test)):
        ids = sorted(int(i) for i in ids)
        out[name] = {"paths": [paths[i] for i in ids], "labels": [int(labels[i]) for i in ids]}
    return out


def get_or_create_split(root, split_path, fractions, seed, max_per_class=None, force=False) -> dict:
    """Load the saved split if it matches the current settings, else create + save it.

    Persisting the split guarantees every model is trained and tested on exactly the
    same images, which is essential for a fair CNN vs transfer-learning comparison.
    """
    root = Path(root)
    split_path = Path(split_path)
    meta = {"root": str(root.resolve()), "seed": seed, "max_per_class": max_per_class,
            "fractions": list(fractions)}
    if split_path.exists() and not force:
        info = json.loads(split_path.read_text())
        if info.get("meta") == meta:
            return info
        print("[data] Existing split.json used different settings -> regenerating it.\n"
              "       Models trained on the old split should be retrained before evaluation.")

    paths, labels, class_names = list_images(root)
    if max_per_class:
        paths, labels = limit_per_class(paths, labels, max_per_class, seed)
    splits = stratified_split(paths, labels, fractions, seed)
    info = {"meta": meta, "class_names": class_names, **splits}
    split_path.parent.mkdir(parents=True, exist_ok=True)
    split_path.write_text(json.dumps(info))
    return info


def compute_class_weights(labels, num_classes: int) -> dict[int, float]:
    """'Balanced' class weights (PlantVillage is imbalanced, e.g. 152 vs 5507 images/class)."""
    classes = np.arange(num_classes)
    present = np.unique(labels)
    weights = compute_class_weight("balanced", classes=present, y=np.asarray(labels))
    out = {int(c): 1.0 for c in classes}
    out.update({int(c): float(w) for c, w in zip(present, weights)})
    return out


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/evaluate.py
"""Evaluate trained models on the held-out test set and compare them.

    python -m src.evaluate                      # every model that has been trained
    python -m src.evaluate --models cnn resnet50

Outputs (under outputs/):
    <model>/metrics.json, classification_report.csv,
    <model>/confusion_matrix.png, <model>/confusion_matrix_normalized.png
    comparison/metrics_comparison.csv / .png, comparison/training_curves_comparison.png
"""
from __future__ import annotations

import argparse
import json
import time

import numpy as np
import pandas as pd
import keras

from . import config as C
from .data_loader import make_dataset
from .metrics import compute_metrics
from .models import (AddPositionEmbs,  # noqa: F401  (registers the custom layer for loading)
                    ResNetPreprocess)
from .plots import (plot_confusion_matrix, plot_history_comparison, plot_metrics_comparison)


def evaluate_model(name: str, split: dict, batch_size: int) -> dict:
    class_names = split["class_names"]
    model = keras.models.load_model(C.model_path(name), compile=False)
    test = split["test"]
    test_ds = make_dataset(split["meta"]["root"], test["paths"], test["labels"], batch_size, training=False)

    start = time.time()
    probs = model.predict(test_ds, verbose=0)
    seconds = time.time() - start
    y_true = np.asarray(test["labels"])
    y_pred = probs.argmax(axis=1)

    result = compute_metrics(y_true, y_pred, class_names, probs)
    result.update(
        model=name,
        params_millions=model.count_params() / 1e6,
        size_mb=C.model_path(name).stat().st_size / 1e6,
        ms_per_image=1000 * seconds / len(y_true),   # includes image decoding + resizing
    )

    out_dir = C.OUTPUTS_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(result["report"]).T.to_csv(out_dir / "classification_report.csv")
    slim = {k: v for k, v in result.items() if k not in ("report", "confusion_matrix")}
    (out_dir / "metrics.json").write_text(json.dumps(slim, indent=2))
    title = C.DISPLAY_NAMES[name]
    plot_confusion_matrix(result["confusion_matrix"], class_names, out_dir / "confusion_matrix.png",
                          normalize=False, title=title)
    plot_confusion_matrix(result["confusion_matrix"], class_names, out_dir / "confusion_matrix_normalized.png",
                          normalize=True, title=title)
    print(f"[eval] {title}: acc={result['accuracy']:.4f} f1_macro={result['f1_macro']:.4f} "
          f"top3={result.get('top3_accuracy', float('nan')):.4f}")
    return result


def build_comparison(results: dict) -> pd.DataFrame:
    rows = []
    for name, r in results.items():
        hist_path = C.OUTPUTS_DIR / name / "history.json"
        hist = json.loads(hist_path.read_text()) if hist_path.exists() else {}
        rows.append({
            "Model": C.DISPLAY_NAMES[name],
            "Test accuracy": round(r["accuracy"], 4),
            "Precision (macro)": round(r["precision_macro"], 4),
            "Recall (macro)": round(r["recall_macro"], 4),
            "F1 (macro)": round(r["f1_macro"], 4),
            "F1 (weighted)": round(r["f1_weighted"], 4),
            "Top-3 accuracy": round(r.get("top3_accuracy", float("nan")), 4),
            "Params (M)": round(r["params_millions"], 2),
            "Size (MB)": round(r["size_mb"], 1),
            "Inference (ms/img)": round(r["ms_per_image"], 1),
            "Epochs run": hist.get("epochs_run"),
            "Train time (min)": round(hist["train_time_sec"] / 60, 1) if "train_time_sec" in hist else None,
        })
    return pd.DataFrame(rows)


def main(argv=None) -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--models", nargs="+", default=list(C.MODEL_NAMES), choices=C.MODEL_NAMES)
    p.add_argument("--batch-size", type=int, default=C.BATCH_SIZE)
    args = p.parse_args(argv)

    if not C.SPLIT_PATH.exists():
        raise SystemExit("outputs/split.json not found - run `python -m src.train` first.")
    split = json.loads(C.SPLIT_PATH.read_text())

    results = {}
    for name in args.models:
        if not C.model_path(name).exists():
            print(f"[eval] skipping {name}: {C.model_path(name)} not found")
            continue
        results[name] = evaluate_model(name, split, args.batch_size)
    if not results:
        raise SystemExit("No trained models found in models/.")

    C.COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
    table = build_comparison(results)
    table.to_csv(C.COMPARISON_DIR / "metrics_comparison.csv", index=False)
    plot_metrics_comparison(results, C.COMPARISON_DIR / "metrics_comparison.png", C.DISPLAY_NAMES)

    histories = {n: json.loads((C.OUTPUTS_DIR / n / "history.json").read_text())
                 for n in results if (C.OUTPUTS_DIR / n / "history.json").exists()}
    if histories:
        plot_history_comparison(histories, C.COMPARISON_DIR / "training_curves_comparison.png", C.DISPLAY_NAMES)

    print("\n" + table.to_string(index=False))
    print(f"\nSaved comparison to {C.COMPARISON_DIR}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/labels.py
"""Turn PlantVillage folder names into human-readable labels (no TensorFlow needed)."""
from __future__ import annotations


def parse_label(raw: str) -> tuple[str, str, bool]:
    """Split a class folder name into (plant, condition, is_healthy).

    Handles the naming variants found in the common PlantVillage releases:
    ``Tomato___Early_blight``, ``Tomato__Target_Spot``, ``Tomato_Early_blight``,
    ``Pepper__bell___Bacterial_spot``, ``Corn_(maize)___Common_rust_``.
    """
    plant, condition = raw, ""
    for sep in ("___", "__", "_"):
        if sep in raw:
            plant, condition = raw.split(sep, 1)
            break
    plant = " ".join(plant.replace("_", " ").split())
    condition = " ".join(condition.replace("_", " ").split())
    if not condition:
        condition = "Unknown"
    condition = condition[0].upper() + condition[1:]
    return plant, condition, "healthy" in condition.lower()


def pretty_label(raw: str) -> str:
    plant, condition, _ = parse_label(raw)
    return f"{plant} - {condition}"


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/metrics.py
"""Evaluation metrics (scikit-learn / numpy only)."""
from __future__ import annotations

import numpy as np
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)


def top_k_accuracy(probs: np.ndarray, y_true: np.ndarray, k: int = 3) -> float:
    k = min(k, probs.shape[1])
    top_k = np.argsort(probs, axis=1)[:, -k:]
    return float(np.mean([y in row for y, row in zip(y_true, top_k)]))


def compute_metrics(y_true, y_pred, class_names, probs=None) -> dict:
    """Overall + per-class metrics and the confusion matrix."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    labels = list(range(len(class_names)))
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(p_macro), "recall_macro": float(r_macro), "f1_macro": float(f_macro),
        "precision_weighted": float(p_w), "recall_weighted": float(r_w), "f1_weighted": float(f_w),
        "num_test_samples": int(len(y_true)),
        "report": classification_report(y_true, y_pred, labels=labels, target_names=class_names,
                                        output_dict=True, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }
    if probs is not None:
        result["top3_accuracy"] = top_k_accuracy(np.asarray(probs), y_true, 3)
    return result


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/models.py
"""Model definitions: a custom CNN and two transfer-learning models.

All models take RAW images (float 0-255, shape 224x224x3). Data augmentation and the
model-specific normalisation are layers *inside* the model:
  * augmentation layers are only active during training,
  * normalisation matches what the ImageNet backbone expects.
"""
from __future__ import annotations

import tensorflow as tf
import keras
from keras import layers

from .config import (IMG_SHAPE, VIT_DROPOUT, VIT_MLP_DIM, VIT_NUM_HEADS,
                    VIT_PATCH_SIZE, VIT_PROJECTION_DIM, VIT_TRANSFORMER_LAYERS)


@keras.saving.register_keras_serializable(package="plantdisease")
class AddPositionEmbs(layers.Layer):
    """Learnable 1D positional embedding added to patch sequences.

    Registered as a serializable layer so .keras files reload cleanly.
    """

    def __init__(self, num_patches: int, projection_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = int(num_patches)
        self.projection_dim = int(projection_dim)

    def build(self, input_shape):
        self.pos_embedding = self.add_weight(
            name="pos_embedding",
            shape=(1, self.num_patches, self.projection_dim),
            initializer=keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        return x + self.pos_embedding

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return cfg


@keras.saving.register_keras_serializable(package="plantdisease")
class ResNetPreprocess(layers.Layer):
    """Equivalent of ``resnet50.preprocess_input``: RGB->BGR and ImageNet mean subtraction.

    Implemented as a registered layer so ``.keras`` files reload without Lambda/unsafe code.
    """

    _MEAN = (103.939, 116.779, 123.68)

    def call(self, x):
        x = tf.reverse(x, axis=[-1])
        return x - tf.constant(self._MEAN, dtype=x.dtype)

    def compute_output_shape(self, input_shape):
        return input_shape


def build_augmentation(name: str = "augmentation") -> keras.Sequential:
    """Random flips, rotation, zoom, shift and contrast (training only)."""
    return keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.15, fill_mode="reflect"),
            layers.RandomZoom(0.15, fill_mode="reflect"),
            layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
            layers.RandomContrast(0.15),
        ],
        name=name,
    )


def build_custom_cnn(num_classes: int, img_shape=IMG_SHAPE) -> keras.Model:
    """Four conv blocks (32-64-128-256) with BatchNorm, pooling and dropout."""
    inputs = keras.Input(shape=img_shape, name="image")
    x = build_augmentation()(inputs)
    x = layers.Rescaling(1.0 / 255, name="rescale")(x)
    for filters in (32, 64, 128, 256):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")


def build_transfer_model(name: str, num_classes: int, pretrained: bool = True,
                         img_shape=IMG_SHAPE, dropout: float = 0.3):
    """ImageNet backbone (frozen) + global pooling + dropout + softmax head.

    Returns ``(model, backbone)`` so the trainer can unfreeze the backbone later.
    """
    weights = "imagenet" if pretrained else None
    if name == "mobilenetv2":
        base = keras.applications.MobileNetV2(include_top=False, weights=weights, input_shape=img_shape)
        preprocess = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_preprocess")  # -> [-1, 1]
    elif name == "resnet50":
        base = keras.applications.ResNet50(include_top=False, weights=weights, input_shape=img_shape)
        preprocess = ResNetPreprocess(name="resnet_preprocess")
    else:
        raise ValueError(f"Unknown transfer model: {name}")

    base.trainable = False
    inputs = keras.Input(shape=img_shape, name="image")
    x = build_augmentation()(inputs)
    x = preprocess(x)
    x = base(x, training=False)          # keep BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name=name), base


def build_vit(
    num_classes: int,
    img_shape=IMG_SHAPE,
    patch_size: int = VIT_PATCH_SIZE,
    projection_dim: int = VIT_PROJECTION_DIM,
    num_heads: int = VIT_NUM_HEADS,
    transformer_layers: int = VIT_TRANSFORMER_LAYERS,
    mlp_dim: int = VIT_MLP_DIM,
    dropout: float = VIT_DROPOUT,
) -> keras.Model:
    """Vision Transformer (ViT) architecture based on Dosovitskiy et al. (2020).

    Splits images into non-overlapping patches, linearly projects them, adds
    learnable positional embeddings, processes them through a sequence of
    Transformer Encoder blocks (Multi-Head Self-Attention + MLP with GELU),
    and classifies via sequence pooling and an MLP prediction head.
    """
    num_patches = (img_shape[0] // patch_size) * (img_shape[1] // patch_size)

    inputs = keras.Input(shape=img_shape, name="image")
    # Data augmentation active during training
    x = build_augmentation(name="vit_augmentation")(inputs)
    # In-model normalization to [0, 1]
    x = layers.Rescaling(1.0 / 255.0, name="rescaling")(x)

    # Patch extraction & linear projection via non-overlapping Conv2D
    x = layers.Conv2D(
        filters=projection_dim,
        kernel_size=patch_size,
        strides=patch_size,
        padding="valid",
        name="patch_conv",
    )(x)
    x = layers.Reshape((num_patches, projection_dim), name="patch_flatten")(x)

    # Positional embedding + dropout
    x = AddPositionEmbs(num_patches=num_patches, projection_dim=projection_dim, name="pos_emb")(x)
    x = layers.Dropout(dropout, name="pos_drop")(x)

    # Transformer Encoder blocks
    for i in range(transformer_layers):
        # Multi-Head Attention block with residual
        norm1 = layers.LayerNormalization(epsilon=1e-6, name=f"ln1_{i}")(x)
        attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim // num_heads,
            dropout=dropout,
            name=f"mha_{i}",
        )(norm1, norm1)
        x = layers.Add(name=f"attn_add_{i}")([x, attn])

        # MLP block with residual
        norm2 = layers.LayerNormalization(epsilon=1e-6, name=f"ln2_{i}")(x)
        mlp = layers.Dense(mlp_dim, activation="gelu", name=f"mlp1_{i}")(norm2)
        mlp = layers.Dropout(dropout, name=f"mlp_drop_{i}")(mlp)
        mlp = layers.Dense(projection_dim, name=f"mlp2_{i}")(mlp)
        mlp = layers.Dropout(dropout, name=f"mlp_out_drop_{i}")(mlp)
        x = layers.Add(name=f"mlp_add_{i}")([x, mlp])

    # Representation head: sequence normalization + pooling
    x = layers.LayerNormalization(epsilon=1e-6, name="head_ln")(x)
    x = layers.GlobalAveragePooling1D(name="seq_pooling")(x)
    x = layers.Dropout(0.2, name="head_dropout")(x)

    # Classification MLP head
    x = layers.Dense(mlp_dim // 2, activation="gelu", name="head_mlp")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    return keras.Model(inputs=inputs, outputs=outputs, name="vit")


def build_model(name: str, num_classes: int, pretrained: bool = True):
    """Factory. Returns ``(model, backbone_or_None)``."""
    if name == "cnn":
        return build_custom_cnn(num_classes), None
    if name == "vit":
        return build_vit(num_classes), None
    return build_transfer_model(name, num_classes, pretrained=pretrained)


def unfreeze_top_layers(base: keras.Model, n_layers: int) -> None:
    """Fine-tuning: make the last ``n_layers`` of the backbone trainable (BatchNorm stays frozen)."""
    base.trainable = True
    for layer in base.layers[:-n_layers]:
        layer.trainable = False
    for layer in base.layers[-n_layers:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False


def compile_model(model: keras.Model, learning_rate: float) -> None:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/plots.py
"""Matplotlib figures: training curves, confusion matrices, model comparison."""
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe
import matplotlib.pyplot as plt
import numpy as np

from .labels import pretty_label

COLORS = {"cnn": "#d95f02", "mobilenetv2": "#1b9e77", "resnet50": "#7570b3", "vit": "#e7298a"}


def _save(fig, path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_history(history: dict, path, title: str) -> None:
    """Accuracy and loss (train vs validation). Dashed line marks the start of fine-tuning."""
    epochs = np.arange(1, len(history["loss"]) + 1)
    boundary = history.get("phase_boundary")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, key, label in ((axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")):
        ax.plot(epochs, history[key], marker="o", label=f"Train {label.lower()}")
        ax.plot(epochs, history[f"val_{key}"], marker="s", label=f"Validation {label.lower()}")
        if boundary:
            ax.axvline(boundary + 0.5, ls="--", color="gray", label="Fine-tuning starts")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(label)
        ax.set_title(f"{title}: {label}")
        ax.grid(alpha=0.3)
        ax.legend()
    _save(fig, path)


def plot_history_comparison(histories: dict, path, display_names: dict | None = None) -> None:
    """Validation accuracy / loss of all models on the same axes."""
    display_names = display_names or {}
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, h in histories.items():
        epochs = np.arange(1, len(h["loss"]) + 1)
        for ax, key in ((axes[0], "val_accuracy"), (axes[1], "val_loss")):
            ax.plot(epochs, h[key], marker="o", color=COLORS.get(name), label=display_names.get(name, name))
    axes[0].set_title("Validation accuracy")
    axes[1].set_title("Validation loss")
    for ax in axes:
        ax.set_xlabel("Epoch (fine-tuning epochs continue the same axis)")
        ax.grid(alpha=0.3)
        ax.legend()
    _save(fig, path)


def plot_confusion_matrix(cm, class_names, path, normalize: bool = False, title: str = "Confusion matrix") -> None:
    cm = np.asarray(cm, dtype=float)
    if normalize:
        cm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    n = len(class_names)
    size = max(6.0, 0.42 * n + 3)
    fig, ax = plt.subplots(figsize=(size, size * 0.9))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    labels = [pretty_label(c) for c in class_names]
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=90, fontsize=max(5, 9 - n // 8))
    ax.set_yticklabels(labels, fontsize=max(5, 9 - n // 8))
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title + (" (row-normalised)" if normalize else ""))
    if n <= 20:  # annotate cells only when readable
        thresh = cm.max() / 2 if cm.max() > 0 else 0.5
        for i in range(n):
            for j in range(n):
                txt = f"{cm[i, j]:.2f}" if normalize else f"{int(cm[i, j])}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                        color="white" if cm[i, j] > thresh else "black")
    _save(fig, path)


def plot_metrics_comparison(table: dict, path, display_names: dict | None = None) -> None:
    """Grouped bars: accuracy / precision / recall / F1 for every model.

    ``table`` maps model name -> metrics dict (as produced by ``compute_metrics``).
    """
    display_names = display_names or {}
    keys = [("accuracy", "Accuracy"), ("precision_macro", "Precision"),
            ("recall_macro", "Recall"), ("f1_macro", "F1 (macro)")]
    names = list(table)
    x = np.arange(len(keys))
    width = 0.8 / max(len(names), 1)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, name in enumerate(names):
        vals = [table[name][k] for k, _ in keys]
        bars = ax.bar(x + i * width - 0.4 + width / 2, vals, width, color=COLORS.get(name),
                      label=display_names.get(name, name))
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.005, f"{v:.3f}", ha="center", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([lbl for _, lbl in keys])
    lo = min(min(table[n][k] for k, _ in keys) for n in names)
    ax.set_ylim(max(0.0, lo - 0.1), 1.03)
    ax.set_title("Test-set metrics: CNN vs transfer learning")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="lower right")
    _save(fig, path)


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/predict.py
"""Single-image inference, used by the Streamlit app and usable from the command line.

    python -m src.predict --image leaf.jpg --model mobilenetv2 --top-k 3
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps
import keras

from . import config as C
from .labels import parse_label
from .models import (AddPositionEmbs,  # noqa: F401  (registers the custom layer for loading)
                    ResNetPreprocess)


def load_class_names(path: Path = C.CLASS_NAMES_PATH) -> list[str]:
    if not Path(path).exists():
        raise FileNotFoundError(f"{path} not found - train a model first (python -m src.train).")
    return json.loads(Path(path).read_text())


class Predictor:
    """Loads a saved ``.keras`` model and classifies PIL images."""

    def __init__(self, model_file: str | Path, class_names: list[str] | None = None):
        self.model = keras.models.load_model(model_file, compile=False)
        self.class_names = class_names or load_class_names()
        out_units = self.model.output_shape[-1]
        if out_units != len(self.class_names):
            raise ValueError(f"Model outputs {out_units} classes but class_names.json lists "
                             f"{len(self.class_names)}. Retrain or restore the matching class_names.json.")

    @staticmethod
    def preprocess(image: Image.Image) -> np.ndarray:
        """Same decode path as training: RGB, bilinear resize to 224x224, raw 0-255 floats."""
        image = ImageOps.exif_transpose(image).convert("RGB")
        arr = tf.image.resize(np.asarray(image, dtype=np.float32), C.IMG_SIZE).numpy()
        return arr[None, ...]

    def predict(self, image: Image.Image, top_k: int = 3) -> list[dict]:
        probs = self.model(self.preprocess(image), training=False).numpy()[0]
        order = np.argsort(probs)[::-1][:top_k]
        results = []
        for i in order:
            plant, condition, healthy = parse_label(self.class_names[i])
            results.append({"class_name": self.class_names[i], "plant": plant, "condition": condition,
                            "healthy": healthy, "confidence": float(probs[i])})
        return results


def main(argv=None) -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--image", required=True)
    p.add_argument("--model", default="mobilenetv2", choices=C.MODEL_NAMES)
    p.add_argument("--top-k", type=int, default=3)
    args = p.parse_args(argv)
    predictor = Predictor(C.model_path(args.model))
    for rank, r in enumerate(predictor.predict(Image.open(args.image), args.top_k), 1):
        print(f"{rank}. {r['plant']} - {r['condition']:<40s} {r['confidence'] * 100:5.1f}%")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/plant_disease_detection/src/train.py
"""Train the custom CNN and/or the transfer-learning models.

Examples
--------
    python -m src.train --model all
    python -m src.train --model mobilenetv2 --max-per-class 300      # quick CPU run
    python -m src.train --model cnn --epochs 30
"""
from __future__ import annotations

import argparse
import json
import time

import keras

from . import config as C
from .data_loader import prepare_data
from .data_utils import compute_class_weights
from .models import build_model, compile_model, unfreeze_top_layers
from .plots import plot_history


def _callbacks(patience: int) -> list:
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=max(1, patience // 2), min_lr=1e-7, verbose=1),
    ]


def _merge(histories) -> dict:
    merged: dict[str, list[float]] = {}
    for h in histories:
        for key, values in h.history.items():
            merged.setdefault(key, []).extend(float(v) for v in values)
    return merged


def train_one(name: str, data: dict, args) -> dict:
    print(f"\n{'=' * 70}\n Training: {C.DISPLAY_NAMES[name]}\n{'=' * 70}")
    keras.utils.set_random_seed(C.SEED)
    num_classes = len(data["class_names"])
    model, base = build_model(name, num_classes, pretrained=not args.no_pretrained)
    model.summary(line_length=100)

    class_weight = None
    if not args.no_class_weight:
        class_weight = compute_class_weights(data["split"]["train"]["labels"], num_classes)

    fit_kwargs = dict(validation_data=data["val_ds"], class_weight=class_weight, verbose=2)
    histories, boundary = [], None
    start = time.time()

    if name in ("cnn", "vit"):
        lr = C.LR_VIT if name == "vit" else C.LR_CNN
        epochs = args.epochs or (C.VIT_EPOCHS if name == "vit" else C.CNN_EPOCHS)
        compile_model(model, lr)
        histories.append(model.fit(data["train_ds"], epochs=epochs,
                                   callbacks=_callbacks(patience=5), **fit_kwargs))
    else:
        # Phase 1: train only the new classification head on the frozen backbone.
        compile_model(model, C.LR_HEAD)
        h1 = model.fit(data["train_ds"], epochs=args.head_epochs, callbacks=_callbacks(patience=3), **fit_kwargs)
        histories.append(h1)
        boundary = len(h1.history["loss"])
        # Phase 2: unfreeze the top backbone layers and fine-tune with a tiny learning rate.
        unfreeze_top_layers(base, args.fine_tune_layers)
        compile_model(model, C.LR_FINE_TUNE)
        histories.append(model.fit(data["train_ds"], epochs=args.finetune_epochs,
                                   callbacks=_callbacks(patience=4), **fit_kwargs))

    train_seconds = time.time() - start
    history = _merge(histories)
    history.update(phase_boundary=boundary, train_time_sec=train_seconds, epochs_run=len(history["loss"]))

    out_dir = C.OUTPUTS_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    C.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    model.save(C.model_path(name))
    (out_dir / "history.json").write_text(json.dumps(history, indent=2))
    plot_history(history, out_dir / "training_curves.png", C.DISPLAY_NAMES[name])
    print(f"[train] saved {C.model_path(name)} | {history['epochs_run']} epochs | {train_seconds / 60:.1f} min")
    return history


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model", default="all", choices=[*C.MODEL_NAMES, "all"])
    p.add_argument("--data-dir", default=str(C.DATA_DIR))
    p.add_argument("--batch-size", type=int, default=C.BATCH_SIZE)
    p.add_argument("--epochs", type=int, default=None, help="Custom CNN epochs (default from config)")
    p.add_argument("--head-epochs", type=int, default=C.HEAD_EPOCHS, help="Transfer learning phase 1 epochs")
    p.add_argument("--finetune-epochs", type=int, default=C.FINE_TUNE_EPOCHS, help="Transfer learning phase 2 epochs")
    p.add_argument("--fine-tune-layers", type=int, default=C.FINE_TUNE_LAYERS)
    p.add_argument("--max-per-class", type=int, default=None, help="Use at most N images per class (fast runs)")
    p.add_argument("--no-pretrained", action="store_true", help="Do not download ImageNet weights (offline smoke test)")
    p.add_argument("--no-class-weight", action="store_true", help="Disable balanced class weights")
    p.add_argument("--resplit", action="store_true", help="Force a new train/val/test split")
    return p.parse_args(argv)


def main(argv=None) -> None:
    args = parse_args(argv)
    data = prepare_data(args.data_dir, args.batch_size, max_per_class=args.max_per_class, resplit=args.resplit)
    names = C.MODEL_NAMES if args.model == "all" else (args.model,)
    for name in names:
        train_one(name, data, args)
    print("\nDone. Next: python -m src.evaluate")


if __name__ == "__main__":
    main()


In [ ]:
# STEP 1: quick test (about 2-3 minutes) to make sure everything works before the long run
%cd /kaggle/working/plant_disease_detection
!python -m src.train --model all --data-dir "{DATA_DIR}" --max-per-class 20 --epochs 1 --head-epochs 1 --finetune-epochs 1 --batch-size 16
!python -m src.evaluate --batch-size 16
for m in ("cnn", "mobilenetv2", "resnet50", "vit"):
    assert os.path.exists(f"models/{m}.keras"), f"Quick test FAILED for {m}: read the error printed above this cell"
print("QUICK TEST PASSED - now running the full training")


In [ ]:
# STEP 2: full training (this is the long part)
LIMIT = f"--max-per-class {MAX_PER_CLASS}" if MAX_PER_CLASS else ""
!rm -rf models outputs
!python -m src.train --model all --data-dir "{DATA_DIR}" {LIMIT} --epochs {CNN_EPOCHS} --head-epochs {HEAD_EPOCHS} --finetune-epochs {FINETUNE_EPOCHS} --batch-size {BATCH_SIZE}
for m in ("cnn", "mobilenetv2", "resnet50", "vit"):
    assert os.path.exists(f"models/{m}.keras"), f"Training FAILED for {m}: read the error printed above this cell"
print("TRAINING DONE")


In [ ]:
# STEP 3: evaluation (graphs, confusion matrices, comparison table)
!python -m src.evaluate --batch-size {BATCH_SIZE}
assert os.path.exists("outputs/comparison/metrics_comparison.csv"), "Evaluation FAILED: read the error printed above this cell"
print("EVALUATION DONE")

In [ ]:
# Package the trained models + graphs for download
import zipfile, pathlib
root = pathlib.Path("/kaggle/working/plant_disease_detection")
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("models", "outputs"):
        for p in (root / folder).rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(root))
    z.writestr("TF_VERSION.txt", "Trained with TensorFlow " + tf.__version__ + "\n")
print("results.zip size (MB):", round(os.path.getsize("/kaggle/working/results.zip") / 1e6, 1))
import pandas as pd
display(pd.read_csv(root / "outputs/comparison/metrics_comparison.csv"))

**Done.** Download `results.zip` from the Output panel (right side, `/kaggle/working`). If your PC's TensorFlow differs from the version printed above and the app fails to load a model, run `pip install tensorflow==<that version>`.